# OMA pipeline (Kaggle) — fig4 / fig5 / fig8 + eval & ablation
Attach your **OMA** dataset (flat `.mat` + `.py` files), then **Run All**.
Outputs land in `/kaggle/working` and are zipped in the last cell.

In [ ]:
import os, shutil, glob, sys
from pathlib import Path

WORK = Path('/kaggle/working'); WORK.mkdir(exist_ok=True)
os.chdir(WORK)

# --- locate the attached dataset (first dir under /kaggle/input) ---
inps = sorted(glob.glob('/kaggle/input/*'))
assert inps, 'No dataset attached! Add your OMA dataset via "Add Data".'
DATA = inps[0]
print('Input dataset:', DATA)

# --- copy everything flat into WORK (search recursively, take basename) ---
need_py  = ['train_dl_v3_easy.py', 'train_dl_oma_maml.py', 'train_noris_oma.py', 'compare_oma_v3_easy.py', 'aggregate_fig5_oma.py', 'aggregate_fig8_oma.py']
need_mat = ['ISAC_RIS_OMA_channels_v3_easy.mat', 'ISAC_RIS_OMA_channels_v3_easy_N16.mat', 'ISAC_RIS_OMA_channels_v3_easy_N32.mat', 'ISAC_RIS_OMA_channels_v3_easy_N64.mat', 'ISAC_RIS_OMA_channels_v3_easy_TEST.mat', 'ISAC_OMA_channels_noris.mat']
found = {}
for f in glob.glob(os.path.join(DATA, '**', '*'), recursive=True):
    b = os.path.basename(f)
    if b in need_py or b in need_mat:
        shutil.copy(f, WORK / b); found[b] = True
missing = [f for f in need_py + need_mat if f not in found]
print('Copied :', sorted(found))
print('MISSING:', missing if missing else 'none')

# --- build the directory tree the sweeps / aggregators expect ---
for d in ['fig4', 'fig5_oma', 'fig8_oma/ris', 'fig8_oma/no_ris', 'no_ris']:
    (WORK / d).mkdir(parents=True, exist_ok=True)
# aggregators locate checkpoints relative to their own folder -> move them in
for agg, dst in [('aggregate_fig5_oma.py', 'fig5_oma'),
                 ('aggregate_fig8_oma.py', 'fig8_oma')]:
    if (WORK / agg).exists():
        shutil.copy(WORK / agg, WORK / dst / agg)
# no-RIS dataset also expected under no_ris/ by the trainer default
nor = 'ISAC_OMA_channels_noris.mat'
if (WORK / nor).exists():
    shutil.copy(WORK / nor, WORK / 'no_ris' / nor)

import torch
print('CUDA:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
EPOCHS = 40          # bump to 60-80 for final-quality curves
BATCH  = 64
LR     = '1e-4'
print('EPOCHS', EPOCHS, 'BATCH', BATCH, 'LR', LR)


## fig4 — train N=8 OMA-MAML policy (also eval + fig5 N=8 checkpoint)

In [ ]:
import subprocess, sys
def run(cmd):
    print('>>', ' '.join(str(c) for c in cmd)); sys.stdout.flush()
    subprocess.run([str(c) for c in cmd], check=True)

run(['python','-u','train_dl_oma_maml.py',
     '--mat','ISAC_RIS_OMA_channels_v3_easy.mat',
     '--epochs',EPOCHS,'--batch',BATCH,'--lr',LR,
     '--w_c','0.7','--w_s','0.3',
     '--out','policy_oma_maml_best.pt',
     '--iter_log','fig4/fig4_oma_iters.npz'])

## fig5 — train N=16, 32, 64

In [ ]:
for N in (16,32,64):
    run(['python','-u','train_dl_oma_maml.py',
         '--mat',f'ISAC_RIS_OMA_channels_v3_easy_N{N}.mat',
         '--epochs',EPOCHS,'--batch',BATCH,'--lr',LR,
         '--out',f'fig5_oma/N{N}_b{BATCH}_lr{LR}.pt'])

## fig8 — P_max sweep: no-RIS + RIS branches

In [ ]:
P_list = [35,37.5,40,42.5,45]
for P in P_list:
    run(['python','-u','train_noris_oma.py',
         '--mat','no_ris/ISAC_OMA_channels_noris.mat',
         '--epochs',60,'--batch',128,'--P_tot_dBm',P,
         '--out',f'fig8_oma/no_ris/noris_oma_P{P}.pt'])
for P in P_list:
    run(['python','-u','train_dl_oma_maml.py',
         '--mat','ISAC_RIS_OMA_channels_v3_easy_N16.mat',
         '--epochs',EPOCHS,'--batch',BATCH,'--lr',LR,'--P_tot_dBm',P,
         '--out',f'fig8_oma/ris/ris_oma_P{P}_b{BATCH}_lr{LR}.pt'])

## eval + ablation -> eval_oma/ (summary.csv x2 + PNGs)

In [ ]:
run(['python','-u','compare_oma_v3_easy.py',
     '--mat','ISAC_RIS_OMA_channels_v3_easy_TEST.mat',
     '--out-dir','eval_oma',
     '--ckpt_maml','policy_oma_maml_best.pt'])

## aggregate -> fig5_oma_results.csv, fig8_oma_results.csv

In [ ]:
run(['python','fig5_oma/aggregate_fig5_oma.py','--batch',BATCH,'--lr',LR,
     '--n8_ckpt','policy_oma_maml_best.pt'])
run(['python','fig8_oma/aggregate_fig8_oma.py','--batch',BATCH,'--lr',LR])

## zip everything for download

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/oma_results','zip','/kaggle/working')
print('Done -> oma_results.zip')
for f in ['eval_oma/summary.csv','eval_oma/ablation/summary.csv',
          'fig5_oma/fig5_oma_results.csv','fig8_oma/fig8_oma_results.csv']:
    print(f, 'OK' if Path(f).exists() else 'MISSING')